# Chaînes-03 — La mesure : le lavage à un acte et la POVM

Troisième carnet de la série [Chaînes](README.md) — distillation du
chapitre 42 du compendium *Chain Theory* (Wissner-Gross, 2026, preprint non relu) :
« Measurement: the wash at one act and the POVM ». Après la règle de Born, l'unitarité et la
relation d'incertitude (Chaînes-02, chapitres 40-41), le fil passe de **la relation entre deux
lectures** à **l'acte d'une lecture** : la mesure, lue ici comme *le lavage à un acte*
(*the wash at one act*).

Le théorème 42.1 énonce la lecture collective (*collective readout*) de la toile jointe de
plusieurs corps — l'objet que la littérature classe sous la **mesure à opérateurs positifs**
(POVM). Comme tous les théorèmes physiques du compendium, il est **conditionnel** :
`Standing: a conditional theorem`, adossé à l'axiome et aux résultats des chapitres 1, 18, 28
et 40, chacun à son propre statut.

Le patron du carnet est celui de la série : **Mesuré** (ce que le code rejoue lui-même),
**Cité** (toute la chaîne dérivative de la source), **Ouvert** (ce que la source ne prouve
pas, listé ligne à ligne — ici la relation de complétude des dix-sept effets, qui **part**).


In [1]:
import itertools
import numpy as np

print("numpy", np.__version__)


numpy 2.4.2


## Le corps de la lecture collective

La plus petite toile jointe de deux corps du chapitre 42 est **deux copies du corps à deux
docks** (le corps de la lecture collective), appariées à un dock. Ce corps compte **neuf
sites** — trois qutrits $A$, $B$, $C$, un site $H$ de dimension 4, trois qubits $a_1$, $a_2$,
$a_3$ et deux qubits de dock $w_1$, $w_2$ — et **dix liens** notés par leurs deux sites
d'extrémité. La **mesure d'un lien** est le produit des dimensions de ses deux extrémités :
9, 12, 8, 4 et 6 aux cinq types $(3,3)$, $(3,4)$, $(2,4)$, $(2,2)$, $(2,3)$.

La source invite elle-même le lecteur à rejouer l'arithmétique : *« A reader reproduces the
four counts with a few lines of arithmetic from the link list above »*. C'est la partie
**Mesuré** la plus directe du chapitre : construire la liste des liens depuis l'énoncé, en
vérifier les cinq mesures, puis retrouver les premières puissances (éq. 59) et les quatre
comptes de paires (éq. 60) — chaque nombre un entier exact.


In [2]:
# Corps a deux docks (ch. 42, p. 118) : neuf sites, dix liens, cinq types.
dims = {"A": 3, "B": 3, "C": 3, "H": 4, "a1": 2, "a2": 2, "a3": 2, "w1": 2, "w2": 2}
liens = [("A", "B"), ("B", "C"), ("A", "C"), ("H", "A"),
         ("H", "a1"), ("H", "a2"), ("H", "a3"), ("a1", "a2"),
         ("B", "w1"), ("C", "w2")]
mesures = {l: dims[l[0]] * dims[l[1]] for l in liens}

# Les cinq types de lien et leur mesure (produit des dimensions des extremites).
types = {}
for l, m in mesures.items():
    cle = tuple(sorted((dims[l[0]], dims[l[1]]), reverse=True))
    types.setdefault(cle, set()).add(l)
for cle in sorted(types, reverse=True):
    print(f"type {cle} : mesure {cle[0]*cle[1]} -> {sorted(types[cle])}")
print("types distincts :", len(types))


type (4, 3) : mesure 12 -> [('H', 'A')]
type (4, 2) : mesure 8 -> [('H', 'a1'), ('H', 'a2'), ('H', 'a3')]
type (3, 3) : mesure 9 -> [('A', 'B'), ('A', 'C'), ('B', 'C')]
type (3, 2) : mesure 6 -> [('B', 'w1'), ('C', 'w2')]
type (2, 2) : mesure 4 -> [('a1', 'a2')]
types distincts : 5


In [3]:
# Eq. 59 : premieres puissances des trois blocs (habillage 9 de la toile jointe a deux copies).
# Liens non frappes d'une copie : les dix liens sauf le lien de dock non apparie Cw2.
habillage = 9
frappe = ("C", "w2")          # les liens de dock non apparies des deux copies sont frappes
non_frappees = [m for l, m in mesures.items() if l != frappe]
S1_copie = habillage * sum(non_frappees)
S1_appariement = habillage * 4    # lien d'appariement w1w1 de type (2,2), mesure 4
S1 = 2 * S1_copie + S1_appariement
print("S1(copie) =", S1_copie, "(attendu 657)")
print("S1(appariement) =", S1_appariement, "(attendu 36)")
print("S1 =", S1, "(attendu 1350)")

# Eq. 60 : les quatre comptes de paires de la lecture collective, lecture frappee.
own = 2 * S1_copie**2
cross = -2 * S1_copie * S1_copie
cross_join = 2 * 2 * S1_appariement * S1_copie
join_join = S1_appariement**2
somme_non_signee = own - cross + cross_join + join_join
print("own =", own, "| cross =", cross, "| cross-join =", cross_join, "| join-join =", join_join)
print("somme non signee =", somme_non_signee, "= 1350^2 =", 1350**2, "->", somme_non_signee == 1350**2)


S1(copie) = 657 (attendu 657)
S1(appariement) = 36 (attendu 36)
S1 = 1350 (attendu 1350)
own = 863298 | cross = -863298 | cross-join = 94608 | join-join = 1296
somme non signee = 1822500 = 1350^2 = 1822500 -> True


### Lecture du résultat

La liste des liens est celle de l'énoncé, mot pour mot : dix liens, cinq types, et la mesure de
chaque type est bien le produit des dimensions de ses extrémités — $(3,3)\to 9$, $(3,4)\to 12$,
$(2,4)\to 8$, $(2,2)\to 4$, $(2,3)\to 6$. Les premières puissances (éq. 59) sortent exactement :
$S_1(\text{copie}) = 657$ (habillage 9 fois les 73 d'un lien frappé en moins), $S_1(\text{appariement})
= 36$, $S_1 = 1350$. Les quatre comptes de paires (éq. 60, figure 17) tombent entiers :
863298, −863298, 94608 et 1296 — les **termes croisés portent le signe moins** — et la somme non
signée $1822500 = 1350^2$ se referme au carré de la première puissance, comme l'énonce la source.
C'est l'arithmétique exacte que le chapitre 42 invitait le lecteur à reproduire.


## L'espace des enregistrements de la paire (éq. 58)

La lecture collective vit sur l'**espace des enregistrements de la paire** — éq. 58, partie
exacte du chapitre. D'après le type de paire à deux qubits lu à la polarité du slot et au
spectre des enregistrements, la source énonce une **garnison de cinquante-quatre** : six
enregistrements fixes par échange, vingt-quatre orbites, et quatre secteurs tenant **24
enregistrements impairs, 18 pairs mixtes, 12 pairs identiques et 3 de clôture** — cinquante-sept
enregistrements en tout, la face de Born **96**, le tout seize et le coupé neuf, avec trente et un
termes à un échange et trois à deux échanges. La somme à deux échanges est un élément exact de
l'anneau :

$$G_2(2,2) = 5216 + 5216\sqrt{2}, \qquad N = 24+18+12+3 = 57, \qquad Z = 96.$$

La garnison elle-même et la valeur $G_2$ sont **Citées** — leur construction remonte aux
chapitres 18 et 28, hors du périmètre de ce carnet. Ce qui se vérifie ici est l'arithmétique
annoncée : les quatre secteurs somment bien à $N = 57$.


In [4]:
# Eq. 58 : les quatre secteurs de l'espace des enregistrements (valeurs CITEES, addition reverifiee).
secteurs = {"impairs": 24, "pairs mixtes": 18, "pairs identiques": 12, "cloture": 3}
N = sum(secteurs.values())
Z = 96                                     # face de Born, citee
G2 = 5216 + 5216 * np.sqrt(2)              # somme a deux echanges, citee
for nom, n in secteurs.items():
    print(f"secteur {nom} : {n}")
print("N =", N, "(cite : 57) ->", N == 57)
print("Z =", Z, "(face de Born, citee)")
print("G_2(2,2) = 5216 + 5216*sqrt(2) ~", round(G2, 1), "(cite)")


secteur impairs : 24
secteur pairs mixtes : 18
secteur pairs identiques : 12
secteur cloture : 3
N = 57 (cite : 57) -> True
Z = 96 (face de Born, citee)
G_2(2,2) = 5216 + 5216*sqrt(2) ~ 12592.5 (cite)


### Lecture du résultat

L'addition se referme : $24 + 18 + 12 + 3 = 57$ enregistrements, comme l'énonce l'éq. 58. Le
reste — la garnison de cinquante-quatre, les six fixes par échange, les vingt-quatre orbites,
la valeur exacte $G_2(2,2) = 5216 + 5216\sqrt 2$, la face de Born 96 — est cité tel quel : sa
construction remonte aux chapitres 18 et 28, et ce carnet ne la rejoue pas. Ce que le code
prouve ici est uniquement que les quatre secteurs annoncés somment au $N$ annoncé — une
vérification d'arithmétique, pas une dérivation.


## Le swap d'intrication : distiller au qudit partagé

Le mécanisme que la source met au centre du théorème 42.1 est la **toile debout** : *« one
stabilizer state with two generators per link »* — et sa distillation : *« distilling at every
shared qudit is the entanglement swap, leaving the far ends a pair on their common support at
every outcome [...] with a Weyl letter as the byproduct »*. La correspondance avec la
littérature est nommée : l'échange d'intrication projette deux systèmes sur une paire
maximalement intriquée et laisse leurs partenaires intriqués à un opérateur de sous-produit
près [22].

C'est rejouable ici, à l'extrémité qubit, en quelques lignes : deux paires de Bell $AB$ et $CD$
dans $\lvert\omega\rangle = (\lvert 00\rangle + \lvert 11\rangle)/\sqrt 2$, une lecture du couple
central $BC$ dans la **base de Bell** — quatre issues, une par lettre de Weyl $W \in \{I, X, Z,
XZ\}$, chacune $\lvert\!\mathrm{bell}\rangle = (I \otimes W)\lvert\omega\rangle$ (les lettres de
Chaînes-02) — et la question : dans quel état tombe le couple éloigné $AD$ ?


In [5]:
# Swap d'intrication a l'extremite qubit : deux paires de Bell, lecture de BC en base de Bell.
ket = np.array([1, 0], dtype=complex)
ket_b = np.array([0, 1], dtype=complex)
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
omega2 = (np.kron(ket, ket) + np.kron(ket_b, ket_b)) / np.sqrt(2)   # |omega> sur une paire

# Quatre issues de la lecture, une par lettre de Weyl : bell_W = (I (x) W)|omega>.
lettres = {"I": I2, "X": X, "Z": Z, "XZ": X @ Z}
bell = {nom: np.kron(I2, W) @ omega2 for nom, W in lettres.items()}
orth = np.allclose(sum(np.outer(v, v.conj()) for v in bell.values()), np.eye(4))
print("les 4 etats de Bell forment une base orthonormee :", orth)

# Etat de depart : |omega>_AB (x) |omega>_CD sur ABCD ; lecture projective de BC.
psi0 = np.kron(omega2, omega2)
print("\nlettre W | proba | AD pur | trace partielle = I/2 | fidelite avec |bell_W>")
for nom, v in bell.items():
    P_BC = np.kron(I2, np.outer(v, v.conj()))        # projecteur sur BC seul
    phi = np.kron(P_BC, I2) @ psi0                   # etat (non normalise) de ABCD
    p = np.vdot(phi, phi).real                       # probabilite de l'issue
    rho = np.outer(phi, phi.conj()) / p              # etat post-lecture, ABCD
    rho_t = rho.reshape(2, 2, 2, 2, 2, 2, 2, 2)      # A,B,C,D,A',B',C',D'
    # trace partielle sur B et C : apparier (B,B') et (C,C') par lettres partagees b et c
    rho_AD = np.einsum("abcdebch->adeh", rho_t).reshape(4, 4)
    purs = abs(np.trace(rho_AD @ rho_AD).real - 1) < 1e-12      # Tr(rho^2) = 1
    rho_A = np.einsum("aded->ae", rho_AD.reshape(2, 2, 2, 2))   # trace partielle sur D
    plat = np.allclose(rho_A, I2 / 2)
    fid = np.vdot(v, rho_AD @ v).real
    print(f"    {nom:>2}   | {p:.3f}  |  {purs}  |         {plat}          | {fid:.12f}")


les 4 etats de Bell forment une base orthonormee : True

lettre W | proba | AD pur | trace partielle = I/2 | fidelite avec |bell_W>
     I   | 0.250  |  True  |         True          | 1.000000000000
     X   | 0.250  |  True  |         True          | 1.000000000000
     Z   | 0.250  |  True  |         True          | 1.000000000000
    XZ   | 0.250  |  True  |         True          | 1.000000000000


### Lecture du résultat

À chacune des quatre issues de la lecture du couple central — une issue par lettre de Weyl — le
couple éloigné $AD$ tombe **exactement** dans l'état de Bell qui porte la même lettre : pureté
$\mathrm{Tr}(\rho^2) = 1$ à $10^{-12}$ près, trace partielle $I/2$ (intrication maximale), et
fidélité $1{,}000000000000$ avec $\lvert\mathrm{bell}_W\rangle$. Chaque issue a la probabilité
$1/4$, et le sous-produit de la distillation est la lettre $W$ elle-même. C'est le geste que le
compendium décrit à chaque qudit partagé (*« with a Weyl letter as the byproduct »*) — rejoué
ici à l'extrémité qubit de la construction. Les deux paires de départ ne se touchaient pas :
l'intrication entre $A$ et $D$ est **créée par l'acte de lecture seul** — c'est le sens de
*the wash at one act*.


## La POVM côté littérature — le référentiel du gloss

Le gloss du chapitre 42 s'appuie sur la définition de la littérature : une **mesure généralisée**
est un ensemble d'opérateurs positifs $E_i$ sommant à l'identité, avec probabilités d'issue
$\mathrm{Tr}(\rho E_i)$ — et toute mesure généralisée se relève, par le théorème de dilatation,
en une mesure projective sur un espace plus grand [74]. C'est le référentiel **exact** contre
lequel la source fait partir sa propre construction : la question n'est pas de savoir si des
opérateurs positifs existent, mais si leur somme **est** l'identité du registre.

Pour voir ce que « complète » veut dire, rien ne vaut une POVM qui l'est : la **trine** — trois
effets $E_k = \tfrac{2}{3}\lvert\psi_k\rangle\langle\psi_k\rvert$ aux kets séparés de 120° sur
l'équateur du Bloch. La somme des effets est l'identité **par construction**, et les probabilités
$\mathrm{Tr}(\rho E_k)$ somment à 1 pour **tout** état — pur ou mixte — ce qui est exactement le
contenu de la relation de complétude.


In [6]:
# La trine : trois effets positifs sur un qubit, complete par construction.
t = 2 * np.pi / 3
ket_k = [np.array([1, np.exp(1j * k * t)]) / np.sqrt(2) for k in range(3)]
effets = [(2 / 3) * np.outer(k, k.conj()) for k in ket_k]

somme = sum(effets)
print("somme des effets == I_2 :", np.allclose(somme, np.eye(2)))
print("hermitiens et positifs :", all(np.allclose(e, e.conj().T) and
      min(np.linalg.eigvalsh(e).real) >= -1e-12 for e in effets))

# Probabilites d'issue Tr(rho E_k) sur trois etats distincts : pur |0>, pur |+>, mixte maximal.
etats = {
    "|0> (pur)": np.outer(ket, ket.conj()),
    "|+> (pur)": np.outer((ket + ket_b) / np.sqrt(2), (ket + ket_b).conj() / np.sqrt(2)),
    "I/2 (mixte maximal)": np.eye(2) / 2,
}
for nom, rho in etats.items():
    probas = [np.trace(rho @ e).real for e in effets]
    print(f"{nom:>18} : probas {[round(p, 6) for p in probas]} somme {round(sum(probas), 9)}")


somme des effets == I_2 : True
hermitiens et positifs : True
         |0> (pur) : probas [np.float64(0.333333), np.float64(0.333333), np.float64(0.333333)] somme 1.0
         |+> (pur) : probas [np.float64(0.666667), np.float64(0.166667), np.float64(0.166667)] somme 1.0
I/2 (mixte maximal) : probas [np.float64(0.333333), np.float64(0.333333), np.float64(0.333333)] somme 1.0


### Lecture du résultat

La trine est bien une POVM **complète** : $\sum_k E_k = I_2$ au sens machine, chaque effet est
hermitien positif, et les probabilités d'issue somment à $1$ — à $10^{-9}$ près — pour les trois
états essayés, pur ou mixte. Aucun état ne peut violer cette normalisation : c'est une identité
d'opérateurs, pas une propriété d'état. Ce sera la ligne de contraste du **Ouvert** ci-dessous :
dans la construction du compendium, la somme des effets **n'est pas** l'identité du registre —
et c'est précisément pourquoi le théorème 42.1 reste conditionnel et le gloss, un gloss.


## Ce que la source ne prouve pas — ligne à ligne

Le chapitre 42 est explicite sur ses propres limites ; le carnet les rapporte telles quelles,
sans les maquiller en résultats établis.

**Le gloss de la mesure généralisée tient en cinq pas, et un d'eux part.** Quatre pas se
résolvent en théorèmes : le lavage à un acte, le placement (la position d'un tenseur dans un
réseau), le taux (*le carré lavé sur le carré de Born*), et le registre de dock comme tout du
support commun. Mais le cinquième — la **complétude** — part : à la toile jointe de l'électron,
les **dix-sept effets**, chacun un rationnel non négatif exact à l'unité de dock, **sument à la
classe croix au dock, tandis que le registre de dock debout est seize**. La relation de
complétude $\sum_i E_i = I$ part à la fois **à la lecture de Born et au compte**. Les effets
sont construits comme opérateurs sur le registre de dock et leur somme est imprimée, mais
**l'identification de cette somme à l'identité du registre part** — contre la trine ci-dessus,
qui la satisfait par construction. C'est la frontière exacte entre le chapitre et la
littérature qu'il invoque [74].

**Les prémices ouvertes, nommées par la source.** Le théorème 42.1 repose sur les résultats
ouverts 18.5, 18.6, 28.3 et 28.4 — chacun à son propre statut, aucun n'est prouvé au sens du
compendium — en plus des théorèmes conditionnels 18.1-18.4, 28.1-28.2, 28.5 et 40.1. La chaîne
de prémices remonte à l'axiome en douze chapitres au plus : **l'axiome lui-même est posé,
jamais dérivé** (Chaînes-01).

**La lecture qui part au census.** À la lecture nommée dans le census, la lecture collective
**part** : la première part est l'étoile à $N = 4$ corps, **refusée à la borne de valence**.
Le census couvre les six toiles jointes que les deux docks admettent à $N = 2, 3, 4$ — la
lecture est **définie** aux six, mais rien ne dit qu'elle survive au-delà de la borne.

**Preprint non relu.** Le compendium est un *working draft* v0.1, non soumis à relecture ;
ses comptes sont certifiés par son propre paquet machine-checké (« 206 objets énumérés »),
pas par la communauté.


## Ce que ce carnet ne prouve pas

Le code ci-dessus **rejoue** des objets finis — une liste de liens, des comptes de paires, un
swap à l'extrémité qubit, une POVM à trois effets — il **ne démontre** ni le théorème 42.1 ni
aucun de ses maillons. La lecture collective aux six toiles jointes (206 liens au total avec le
corps simple), les **3412 swaps à 1100 qudits partagés sur dix triplets distincts**, le compte
de rondes égal à la profondeur aux **64 toiles**, la sortie comme trace normalisée aux **309
lectures**, le second moteur de reproduction, les contretypes et contrôles : tout cela est
**Cité**, vérifié par la source à ses propres moteurs, jamais rejoué ici. Les dix-sept effets
de l'électron ne sont pas non plus rejoués : leurs valeurs ne sont pas imprimées au chapitre,
seul leur statut l'est (chacun un rationnel non négatif exact, la somme partant de seize).
L'arithmétique que le carnet vérifie est celle que la source invite explicitement le lecteur à
reprendre — rien de plus.


## Croisements

- **Chaînes-02** : le théorème 40.1 (le pli plat) est prémice directe du théorème 42.1, et les
  lettres de Weyl y sont construites — le sous-produit du swap d'ici est l'une d'elles.
- **Chaînes-01** : l'axiome d'intrication, posé et jamais dérivé, tête de la chaîne de douze
  chapitres.
- **Jauge-01/02** : les lois de conservation, elles, sont **théorèmes** (non conditionnels) à la
  source — la mesure est conditionnelle, la conservation est dérivée.
- **Chapitre 43** (indiscernabilité et signe spin-statistique par le twirl) et **chapitre 44**
  (l'énergie comme opérateur de première puissance, éq. 64-67) : candidats naturels pour la
  suite de la série.

## Références

- T. Wissner-Gross, *Chain Theory Compendium* (v0.1 working draft, 2026), chapitre 42
  « Measurement: the wash at one act and the POVM », p. 116-119 — preprint non relu, archivé
  au gisement de la bibliographie, lu avant écriture.
- Références nommées par la source : [22] échange d'intrication avec sous-produit ;
  [74] théorème de dilatation (mesure généralisée ↔ projective sur un espace plus grand) ;
  [75] calcul fondé sur la mesure (carnets de stabilisateur, profondeur = rondes).


## Exercices

### Exercice 1 — Symétrie des docks

La source frappe le lien de dock non apparié $Cw_2$ de chaque copie. Les deux docks $w_1$ et
$w_2$ tiennent des liens de même type $(2,3)$ et même mesure. Vérifier que frapper $Bw_1$ et
d'apparier les seconds docks ($w_2w_2$) donne **les mêmes** premières puissances et les mêmes
quatre comptes de paires — la symétrie des deux docks de la lecture collective.


In [7]:
# Exercice 1 : recalculer eq. 59-60 en frappant Bw1 (au lieu de Cw2) et en apparieant w2w2.
# Etape 1 : construire la liste des liens non frappes avec Bw1 frappe.
# Etape 2 : calculer S1(copie), S1(appariement) et S1.
# Etape 3 : deriver les quatre comptes et comparer a own/cross/cross-join/join-join.
result = None  # TODO etudiant
print("Exercice a completer")


Exercice a completer


### Exercice 2 — Le dictionnaire du sous-produit, lettre par lettre

La table du swap montre la fidélité $1$ pour chaque lettre, mais le sous-produit de la
littérature est une **correspondance** : l'issue $W$ lue sur $BC$ produit l'état
$\lvert\mathrm{bell}_{\bar W}\rangle$ sur $AD$ pour une certaine conjugaison $\bar W$. Reconstruire
le dictionnaire complet : pour chaque lettre $W \in \{I, X, Z, XZ\}$, calculer la fidélité de
$\rho_{AD}$ avec les **quatre** états de Bell, et identifier lequel est sélectionné.

*Indice : comparer $\langle\mathrm{bell}_{W'}|\rho_{AD}|\mathrm{bell}_{W'}\rangle$ sur les
quatre valeurs de $W'$ — la table du carnet s'arrête à la diagonale.*


In [8]:
# Exercice 2 : table de fidelite 4x4 -- lignes = issue lue W sur BC, colonnes = etat W' teste sur AD.
# Etape 1 : boucler sur les quatre issues W et recalculer rho_AD comme dans la cellule du swap.
# Etape 2 : pour chaque W, evaluer la fidelite avec les quatre etats bell[W'].
# Etape 3 : imprimer la table et relever la lettre selectionnee sur chaque ligne.
result = None  # TODO etudiant
print("Exercice a completer")


Exercice a completer


### Exercice 3 — Une POVM qui n'est pas complète

Construire trois effets à partir des kets de la trine mais avec un poids $\alpha$ au lieu de
$\tfrac23$ : $E_k = \alpha\,\lvert\psi_k\rangle\langle\psi_k\rvert$. Trouver la valeur de
$\alpha$ qui rend la mesure complète, puis montrer que pour tout autre $\alpha$ l'écart
$\sum_k \mathrm{Tr}(\rho E_k) - 1$ est **le même pour tous les états** — une identité
d'opérateurs, pas une propriété d'état. Mesurer cet écart sur $\lvert 0\rangle$ et sur $I/2$
pour $\alpha = \tfrac12$ et $\alpha = \tfrac{1}{3}$ : c'est l'écart de complétude, en petit —
celui que la source constate à l'échelle du registre de dock.


In [9]:
# Exercice 3 : trine a poids alpha -- mesurer l'ecart de completude pour alpha != 2/3.
# Etape 1 : trouver alpha tel que sum(alpha |psi_k><psi_k|) == I_2 (somme des projecteurs d'abord).
# Etape 2 : pour alpha = 1/2 et 1/3, calculer l'ecart sum_k Tr(rho E_k) - 1 sur |0> et I/2.
# Etape 3 : verifier que l'ecart est independant de l'etat (identite d'operateurs) et l'interpreter.
result = None  # TODO etudiant
print("Exercice a completer")


Exercice a completer


## À retenir

1. **La mesure est ici le lavage à un acte** : le théorème 42.1 énonce la lecture collective de
   la toile jointe de plusieurs corps, l'objet que la littérature classe sous la POVM — et il
   est **conditionnel**, comme toute la physique du compendium.
2. **L'arithmétique exacte est vérifiable** : corps à neuf sites et dix liens, $S_1 = 1350$, et
   les quatre comptes de paires 863298, $-863298$, 94608, 1296 refermés par
   $1350^2 = 1\,822\,500$ — l'invitation explicite de la source, honorée.
3. **Le mécanisme est le swap** : à chaque qudit partagé, distiller = échanger l'intrication,
   le couple éloigné tombe sur une paire maximalement intriquée **à chaque issue**, avec une
   lettre de Weyl en sous-produit — rejoué ici en fidélité exacte aux quatre issues.
4. **La complétude est la frontière** : la trine satisfait $\sum_i E_i = I$ par construction ;
   les dix-sept effets de la toile de l'électron sument à la classe croix, pas à seize — le
   registre debout. Le gloss de la mesure généralisée tient en cinq pas, et c'est le
   cinquième qui part.
5. **Rien de tout cela n'est établi au sens de la communauté** : prémices ouvertes 18.5, 18.6,
   28.3, 28.4 en amont, étoile refusée à la borne de valence à $N=4$, preprint non relu.


## Vérifiez votre compréhension

1. Quels sont les cinq types de lien du corps à deux docks, et quelle est la mesure de chacun ?
2. Dans l'équation 60, deux des quatre comptes de paires sont négatifs ; pourquoi la somme
   « non signée » se referme-t-elle pourtant sur $1350^2$ ?
3. Après la lecture du couple central $BC$, que valent la trace partielle et la fidélité de
   l'état de $AD$ ? Que disent ces deux nombres sur l'intrication créée par la lecture ?
4. Pourquoi la trine est-elle une POVM complète, alors que la construction du compendium ne
   l'est pas ?

<details>
<summary>Réponses</summary>

1. $(3,3) \to 9$, $(3,4) \to 12$, $(2,4) \to 8$, $(2,2) \to 4$, $(2,3) \to 6$ : la mesure d'un
   lien est le produit des dimensions de ses deux extrémités.
2. Parce que les termes croisés portent le signe moins et que la somme les **retranche** :
   $863298 - (-863298) + 94608 + 1296 = 1\,822\,500 = 1350^2$.
3. Trace partielle $I/2$ et fidélité $1{,}000000000000$ avec $\lvert\mathrm{bell}_W\rangle$ :
   $AD$ est dans une paire maximalement intriquée, alors que les deux paires de départ ne se
   touchaient pas. L'intrication entre $A$ et $D$ est créée par l'acte de lecture seul.
4. La trine somme à l'identité **par construction** ($\sum_k E_k = I_2$), donc ses probabilités
   somment à $1$ pour tout état. Dans la construction de la source, les dix-sept effets de la
   toile de l'électron sument à la classe croix, pas à seize : la somme des effets n'est pas
   l'identité du registre, et c'est pourquoi le théorème 42.1 reste conditionnel.

</details>


## Pour aller plus loin

[Chaînes-04 — Indiscernabilité et signe spin-statistique](Chaines-04-Indiscernabilite-Spin-Statistique-Python.ipynb)
prend la suite du fil : l'indiscernabilité et le signe spin-statistique par le twirl (chapitre 43
du compendium), puis l'énergie comme opérateur de première puissance (chapitre 44). Le geste de
ce carnet y revient : ce que la source dérive et ce qu'elle suppose y sont séparés de la même
manière.

Pour revenir en amont, [Chaînes-02 — la règle de Born, l'unitarité et l'incertitude](Chaines-02-Born-Unitarite-Incertitude-Python.ipynb)
construit les lettres de Weyl qui servent ici de sous-produit au swap, et [Chaînes-01 — l'axiome
d'intrication](Chaines-01-Axiome-Intrication-Constante-Structure-Fine-Python.ipynb) pose l'axiome
dont tout le fil descend.

Hors de la série, [Jauge-01 — le théorème de Noether](../Jauge/Jauge-01-Theoreme-Noether-Python.ipynb) offre le contraste
le plus utile : les lois de conservation y sont des théorèmes, non conditionnels, là où la mesure
est ici conditionnelle à ses prémisses ouvertes. Le chapitre 42 s'appuie enfin sur le théorème de
dilatation de la mesure généralisée [74] et sur les carnets de stabilisateur [75] : deux lectures
qui prolongent la POVM côté littérature.
